# Speech Recognition (ASR): Acoustic Modeling, CTC Loss & WER/CER Evaluation

This interactive lab covers:
1. **The Alignment Problem in ASR**: Why audio frame sequences ($T \approx 500$) vastly outnumber text characters ($U \approx 40$).
2. **Acoustic Encoder Architecture**: 1D temporal downsampling convs and Bidirectional RNN sequence modeling.
3. **Connectionist Temporal Classification (CTC)**: Marginalizing over all valid latent alignments via dynamic programming.
4. **Greedy CTC Decoding**: Collapsing repeating frames and eliminating blank tokens.
5. **Levenshtein Distance & Error Metrics**: Computing Word Error Rate (WER) and Character Error Rate (CER).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('05-speech-audio/speech-recognition-asr/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
import torch.nn as nn
from ctc_engine import (
    MiniAcousticEncoder,
    ctc_greedy_decode,
    compute_wer,
    compute_cer,
)

print('Speech Recognition (ASR) engine loaded successfully!')

## 1. Acoustic Model & CTC Loss Computation

We instantiate an acoustic model predicting 28 character tokens (Blank + 26 alphabet + Space) from 40-channel log-Mel spectrograms.

In [ ]:
torch.manual_seed(42)
vocab_size = 28  # 0 is CTC blank '_'
acoustic_model = MiniAcousticEncoder(in_features=40, hidden_dim=32, num_classes=vocab_size)

# Synthetic batch of 2 audio utterances: (B=2, F=40, T=30)
audio_feats = torch.randn(2, 40, 30)
log_probs = acoustic_model(audio_feats)  # Shape: (T_out=15, B=2, num_classes=28)

# Target transcripts: 'cat' -> [3, 1, 20], 'dog' -> [4, 15, 7]
targets = torch.tensor([[3, 1, 20], [4, 15, 7]])
input_lengths = torch.tensor([15, 15])
target_lengths = torch.tensor([3, 3])

ctc_loss = nn.CTCLoss(blank=0)
loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)

print(f'Input audio frames:             {audio_feats.shape[-1]}')
print(f'Downsampled acoustic time steps: {log_probs.shape[0]}')
print(f'CTC Loss value:                 {loss.item():.4f}')

## 2. CTC Greedy Best-Path Decoding

Demonstrating token collapse and blank elimination on raw alignment frames.

In [ ]:
# Synthetic alignment path for utterance: ['h', 'h', 'e', '_', 'l', 'l', '_', 'l', 'o'] -> 'hello'
synthetic_path = [8, 8, 5, 0, 12, 12, 0, 12, 15]
toy_logits = torch.zeros(len(synthetic_path), 1, vocab_size)
for t, token in enumerate(synthetic_path):
    toy_logits[t, 0, token] = 10.0

decoded_tokens = ctc_greedy_decode(toy_logits, blank_idx=0)
print(f'Frame alignment path: {synthetic_path}')
print(f'Decoded Token IDs:    {decoded_tokens[0]}')
assert decoded_tokens[0] == [8, 5, 12, 12, 15]  # Successfully preserves double 'l'!

## 3. Evaluation: Word Error Rate (WER) & Character Error Rate (CER)

Computing Levenshtein edit distance between ground truth and ASR hypotheses.

In [ ]:
ref_sentence = 'artificial intelligence and deep learning'
hyp_sentence = 'artificial intelligent and deep learning'

wer = compute_wer(ref_sentence, hyp_sentence)
cer = compute_cer(ref_sentence, hyp_sentence)

print(f'Reference:  "{ref_sentence}"')
print(f'Hypothesis: "{hyp_sentence}"')
print(f'Word Error Rate (WER):      {wer * 100:.2f}% (1 substitution / 5 words)')
print(f'Character Error Rate (CER): {cer * 100:.2f}%')